# Дообучение RuBERT-conversational для классификации спам-сообщений

Ноутбук выполняет дообучение модели `DeepPavlov/rubert-base-cased-conversational` (180M параметров, обучена на чатах и соцсетях) на собранном датасете.

Особенности обучения:
- **FP16** — смешанная точность для ускорения обучения на GPU
- **FocalLoss** — функция потерь, фокусирующаяся на сложных примерах
- **EarlyStopping** — ранняя остановка при отсутствии улучшения F1

Обучение выполняется на трёх вариантах текста:
- исходный текст (raw)
- нормализованный текст (normalized)
- предобработанный текст (preprocessed)

Фильтрация: остаются только сообщения, содержащие преимущественно кириллицу.

## Импорт необходимых библиотек

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.transformer import (
    FocalLossTrainer,
    tokenize_function,
    compute_metrics,
    is_mostly_cyrillic,
    get_training_args,
    get_model_config,
    prepare_text_variants,
    benchmark_cpu_inference,
)
from src.config import PROCESSED_DIR, MODELS_DIR

## Чтение обработанного датасета

Загружается `preprocessed.csv` из `data/processed/`.

In [3]:
df = pd.read_csv(PROCESSED_DIR / 'preprocessed.csv', index_col=0)

## Подготовка вариантов текста

Из датасета выделяются три варианта текста для дообучения:
- **raw** — исходный текст (колонка `text`)
- **normalized** — Unicode-нормализация и удаление HTML-тегов
- **preprocessed** — полная предобработка (колонка `text_preprocessed`)

Фильтрация по кириллице: остаются только сообщения, где доля кириллических символов превышает 30%.

In [4]:
variants = prepare_text_variants(df)

ru_variants = {}
for name, vdf in variants.items():
    mask = vdf['text'].apply(lambda t: is_mostly_cyrillic(str(t)))
    ru_variants[name] = vdf[mask].reset_index(drop=True)
    print(f'{name}: {len(ru_variants[name])} сообщений')

raw: 72342 сообщений
normalized: 72355 сообщений
preprocessed: 72503 сообщений


## Дообучение RuBERT-conversational

Модель: `DeepPavlov/rubert-base-cased-conversational`.
Количество классов: 2 (ham=0, spam=1).

### Загрузка модели и токенайзера

In [5]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased-conversational'
model_config = get_model_config(MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Подготовка датасетов и токенизация

Для каждого варианта текста создаётся отдельный train/test сплит (80/20) и токенизируется.

In [5]:
tokenized = {}
for name, vdf in ru_variants.items():
    ds = Dataset.from_pandas(vdf)
    split = ds.train_test_split(test_size=0.2, seed=42)
    train_ds = split['train']
    test_ds = split['test']

    train_tok = train_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )
    test_tok = test_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )

    tokenized[name] = {'train': train_tok, 'test': test_tok}
    print(f'{name}: train={{len(train_tok)}}, test={{len(test_tok)}}')

Map:   0%|          | 0/57873 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Map:   0%|          | 0/14469 [00:00<?, ? examples/s]

raw: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/57884 [00:00<?, ? examples/s]

Map:   0%|          | 0/14471 [00:00<?, ? examples/s]

normalized: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/58002 [00:00<?, ? examples/s]

Map:   0%|          | 0/14501 [00:00<?, ? examples/s]

preprocessed: train={len(train_tok)}, test={len(test_tok)}


### Создание trainer'ов

Используется `FocalLossTrainer`.
Для каждого варианта текста создаётся отдельный trainer.

In [6]:
model_cfg = {
    'learning_rate': 2e-5,
    'epochs': 5,
    'batch_size': 16,
    'fp16': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
}

trainers = {}
for name in tokenized:
    output_dir = f'finetuned_rubert_conv_{name}'

    training_args = get_training_args(
        output_dir=output_dir,
        learning_rate=model_cfg['learning_rate'],
        num_train_epochs=model_cfg['epochs'],
        per_device_train_batch_size=model_cfg['batch_size'],
        per_device_eval_batch_size=model_cfg['batch_size'],
        max_length=model_config['max_length'],
        fp16=model_cfg['fp16'],
        gradient_checkpointing=model_config['gradient_checkpointing'],
    )

    trainer = FocalLossTrainer(
        focal_alpha=model_cfg['focal_alpha'],
        focal_gamma=model_cfg['focal_gamma'],
        model=model,
        args=training_args,
        train_dataset=tokenized[name]['train'],
        eval_dataset=tokenized[name]['test'],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainers[name] = trainer

### Запуск дообучения

Обучение выполняется на GPU (если доступно).

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
model.to(device)

Устройство: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

#### На исходных текстах (raw)

In [8]:
trainers['raw'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001609,0.995024,0.987504,0.985452,0.989565
2,No log,0.001432,0.995853,0.989492,0.996473,0.982609
3,No log,0.001646,0.995024,0.987548,0.982112,0.993043
4,No log,0.002491,0.995922,0.989644,0.998937,0.980522
5,No log,0.001972,0.997028,0.992486,0.997191,0.987826


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18090, training_loss=0.0010657934143505023, metrics={'train_runtime': 832.0679, 'train_samples_per_second': 347.766, 'train_steps_per_second': 21.741, 'total_flos': 3.80675652671232e+16, 'train_loss': 0.0010657934143505023, 'epoch': 5.0})

In [9]:
trainers['raw'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001972,5,0.997028,0.992486,0.997191,0.987826


{'eval_loss': 0.00197199615649879,
 'eval_accuracy': 0.9970281291036008,
 'eval_f1': 0.9924864581513192,
 'eval_precision': 0.9971910112359551,
 'eval_recall': 0.9878260869565217}

Сохранение модели.

In [10]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_conv_raw')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_conv_raw')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_conv_raw/tokenizer_config.json',
 '/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_conv_raw/tokenizer.json')

#### На нормализованных текстах (normalized)

In [11]:
trainers['normalized'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.000769,0.997236,0.992898,0.988685,0.997147
2,No log,0.001581,0.997167,0.992682,0.993569,0.991797
3,No log,0.002037,0.996545,0.991106,0.988644,0.993581


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10854, training_loss=0.0005181532923591407, metrics={'train_runtime': 497.8849, 'train_samples_per_second': 581.299, 'train_steps_per_second': 36.334, 'total_flos': 2.284488049268736e+16, 'train_loss': 0.0005181532923591407, 'epoch': 3.0})

In [12]:
trainers['normalized'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.000769,3,0.997236,0.992898,0.988685,0.997147


{'eval_loss': 0.0007689931662753224,
 'eval_accuracy': 0.9972358510123696,
 'eval_f1': 0.9928977272727273,
 'eval_precision': 0.9886845827439887,
 'eval_recall': 0.9971469329529244}

Сохранение модели.

In [13]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_conv_norm')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_conv_norm')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_conv_norm/tokenizer_config.json',
 '/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_conv_norm/tokenizer.json')

#### На предобработанных текстах (preprocessed)

In [14]:
trainers['preprocessed'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.000951,0.996966,0.992226,0.996805,0.987689
2,No log,0.001063,0.997724,0.994193,0.994718,0.993669
3,No log,0.001138,0.997655,0.994027,0.992980,0.995076
4,No log,0.001008,0.997862,0.994547,0.994722,0.994372
5,No log,0.001135,0.997724,0.994181,0.996818,0.991558


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18130, training_loss=0.00047031871797758375, metrics={'train_runtime': 833.8482, 'train_samples_per_second': 347.797, 'train_steps_per_second': 21.743, 'total_flos': 3.81524185824768e+16, 'train_loss': 0.00047031871797758375, 'epoch': 5.0})

In [15]:
trainers['preprocessed'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001008,5,0.997862,0.994547,0.994722,0.994372


{'eval_loss': 0.001008058781735599,
 'eval_accuracy': 0.9978622163988691,
 'eval_f1': 0.994547053649956,
 'eval_precision': 0.9947220267417312,
 'eval_recall': 0.9943721421034118}

Сохранение модели.

In [16]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_conv_p')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_conv_p')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_conv_p/tokenizer_config.json',
 '/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_conv_p/tokenizer.json')

## Замер CPU-инференса

Замеряется latency инференса на CPU для оценки пригодности модели к развёртыванию на сервере без GPU.

In [6]:
test_messages = [
    "Это честное сообщение от пользователя.",
    "🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎",
    "Зарабатывай миллионы **онлайн** прямо сейчас!",
    "Работа на дому, легкий доход. Пиши в личку!",
    "Привет! Как дела? У меня всё отлично.",
    "steam gift 50$ - steamcommunity.com/gift-card/pay/50\n@everyone",
    "Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!",
    "Как же надоели эти сообщения про казино",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: stankin.ru",
    "3-4 часа и 8 тысяч твои!  Пиши  https://t.me/rasmuswork1",
]

sample_texts = test_messages * 10
suffix_map = {'raw': 'raw', 'normalized': 'norm', 'preprocessed': 'p'}
cpu_results = {}
for name, suffix in suffix_map.items():
    model_path = str(MODELS_DIR / f'finetuned_rubert_conv_{suffix}')
    m = AutoModelForSequenceClassification.from_pretrained(model_path)
    tok = AutoTokenizer.from_pretrained(model_path)
    cpu_results[name] = benchmark_cpu_inference(m, tok, sample_texts, max_length=model_config['max_length'])
    print(f'{name}: {cpu_results[name]}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

raw: {'avg_ms_per_msg': 14.0, 'p95_ms_per_msg': 23.46, 'throughput_msgs_per_sec': 71.44}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

normalized: {'avg_ms_per_msg': 12.75, 'p95_ms_per_msg': 12.84, 'throughput_msgs_per_sec': 78.41}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

preprocessed: {'avg_ms_per_msg': 12.61, 'p95_ms_per_msg': 12.59, 'throughput_msgs_per_sec': 79.33}
